# Gesture Classifier Training Pipeline

## Introduction to Computer Vision - Final Project

This notebook trains a gesture classifier for the Fruit Ninja game.
The classifier recognizes hand gestures (slash, idle, grab, open_palm) from MediaPipe hand landmark features.

### Pipeline Overview
1. Load gesture data collected via `collect_gesture_data.py`
2. Preprocess and split data (70/15/15 train/val/test)
3. Train an MLP neural network classifier using PyTorch
4. Evaluate with accuracy, confusion matrix, and classification report
5. Save trained model for game integration

### Model Architecture
- **Input**: 126 features (21 landmarks × 3 coords × 2 [position + velocity])
- **Hidden layers**: 128 → 64 → 32 neurons with ReLU activation and dropout
- **Output**: 4 gesture classes (slash, idle, grab, open_palm)
- **Why MLP on landmarks**: MediaPipe already extracts hand landmarks as structured features. A simple MLP on these features is more efficient than a CNN on raw images, and the model directly drives the game mechanics.

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
)
import matplotlib.pyplot as plt
import os
import joblib

print(f"PyTorch version: {torch.__version__}")
print(f"MPS available: {torch.backends.mps.is_available()}")
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Load and Explore Data

The dataset was collected using `collect_gesture_data.py`, which captures MediaPipe hand landmarks from webcam video. Each sample contains:
- 63 position features: 21 hand landmarks × 3 coordinates (x, y, z), normalized relative to the wrist
- 63 velocity features: rate of change of each landmark coordinate between frames
- 1 label: the gesture class (slash, idle, grab, open_palm)

In [ ]:
# Load collected gesture data
data_path = os.path.join(os.path.dirname(os.path.abspath('.')), 'gesture_classifier', 'gesture_data.csv')
if not os.path.exists(data_path):
    data_path = 'gesture_data.csv'  # fallback: same directory
if not os.path.exists(data_path):
    raise FileNotFoundError(
        "gesture_data.csv not found. Run collect_gesture_data.py first to record gesture samples."
    )

df = pd.read_csv(data_path)
print(f"Dataset shape: {df.shape}")
print(f"\nSamples per class:")
print(df['label'].value_counts())
print(f"\nFeature columns: {df.shape[1] - 1}")
df.head()

In [ ]:
# Visualize class distribution
fig, ax = plt.subplots(figsize=(8, 4))
df['label'].value_counts().plot(kind='bar', ax=ax, color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4'])
ax.set_title('Gesture Class Distribution')
ax.set_xlabel('Gesture')
ax.set_ylabel('Number of Samples')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150)
plt.show()

## 2. Data Preprocessing

Steps:
1. Separate features (X) and labels (y)
2. Encode labels as integers
3. Standardize features (zero mean, unit variance)
4. Split into train (70%), validation (15%), and test (15%) sets

In [ ]:
# Separate features and labels
X = df.drop('label', axis=1).values.astype(np.float32)
y = df['label'].values

# Encode labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
class_names = label_encoder.classes_
print(f"Classes: {class_names}")
print(f"Encoded: {dict(zip(class_names, range(len(class_names))))}")

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split: 70% train, 15% val, 15% test
X_train, X_temp, y_train, y_temp = train_test_split(
    X_scaled, y_encoded, test_size=0.30, random_state=42, stratify=y_encoded
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"\nTrain: {X_train.shape[0]} samples")
print(f"Val:   {X_val.shape[0]} samples")
print(f"Test:  {X_test.shape[0]} samples")

In [ ]:
# Convert to PyTorch tensors
train_dataset = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.long),
)
val_dataset = TensorDataset(
    torch.tensor(X_val, dtype=torch.float32),
    torch.tensor(y_val, dtype=torch.long),
)
test_dataset = TensorDataset(
    torch.tensor(X_test, dtype=torch.float32),
    torch.tensor(y_test, dtype=torch.long),
)

# DataLoaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Batch size: {batch_size}")
print(f"Train batches: {len(train_loader)}")

## 3. Model Definition

We use a Multi-Layer Perceptron (MLP) because:
- MediaPipe already extracts structured landmark features (not raw pixels)
- The input is a fixed-size vector (126 features), not an image
- An MLP is simpler, faster to train, and sufficient for this task
- The model directly maps hand pose + motion to game actions

Architecture: Input(126) → Linear(128) → ReLU → Dropout(0.3) → Linear(64) → ReLU → Dropout(0.3) → Linear(32) → ReLU → Linear(4)

In [ ]:
class GestureClassifier(nn.Module):
    """MLP classifier for hand gesture recognition from MediaPipe landmarks."""

    def __init__(self, input_size, num_classes, dropout=0.3):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, num_classes),
        )

    def forward(self, x):
        return self.network(x)


input_size = X_train.shape[1]  # 126 features
num_classes = len(class_names)  # 4 gestures

model = GestureClassifier(input_size, num_classes).to(device)
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")

## 4. Training

Training configuration:
- **Optimizer**: Adam (lr=0.001)
- **Loss**: CrossEntropyLoss
- **Epochs**: 100 (with early stopping, patience=10)
- **Device**: Apple M3 Pro with MPS acceleration

In [ ]:
# Training setup
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
num_epochs = 100
patience = 10  # early stopping patience

# Training history
history = {
    'train_loss': [], 'val_loss': [],
    'train_acc': [], 'val_acc': [],
}

best_val_acc = 0.0
best_model_state = None
patience_counter = 0

for epoch in range(num_epochs):
    # Training phase
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

    train_loss /= train_total
    train_acc = train_correct / train_total

    # Validation phase
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_loss /= val_total
    val_acc = val_correct / val_total

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    # Early stopping check
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_state = model.state_dict().copy()
        patience_counter = 0
    else:
        patience_counter += 1

    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}/{num_epochs} | "
              f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

    if patience_counter >= patience:
        print(f"\nEarly stopping at epoch {epoch+1} (best val acc: {best_val_acc:.4f})")
        break

# Restore best model
if best_model_state:
    model.load_state_dict(best_model_state)
    print(f"\nRestored best model with val acc: {best_val_acc:.4f}")

## 5. Training Curves

Visualize the training progress to check for overfitting or underfitting.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
ax1.plot(history['train_loss'], label='Train Loss', color='#FF6B6B')
ax1.plot(history['val_loss'], label='Val Loss', color='#4ECDC4')
ax1.set_title('Training & Validation Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy curves
ax2.plot(history['train_acc'], label='Train Acc', color='#FF6B6B')
ax2.plot(history['val_acc'], label='Val Acc', color='#4ECDC4')
ax2.set_title('Training & Validation Accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

## 6. Evaluation on Test Set

Evaluate the trained model on the held-out test set (15% of data, never seen during training).

In [ ]:
# Evaluate on test set
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

test_accuracy = accuracy_score(all_labels, all_preds)
print(f"Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.1f}%)")
print(f"\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=class_names))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(ax=ax, cmap='Blues', values_format='d')
ax.set_title(f'Gesture Classifier Confusion Matrix\nTest Accuracy: {test_accuracy:.1%}')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

## 7. Per-Class Accuracy Analysis

Examine which gestures are easiest/hardest to classify and common misclassifications.

In [ ]:
# Per-class accuracy
fig, ax = plt.subplots(figsize=(8, 4))
per_class_acc = cm.diagonal() / cm.sum(axis=1)
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4']
bars = ax.bar(class_names, per_class_acc, color=colors[:len(class_names)])

for bar, acc in zip(bars, per_class_acc):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f'{acc:.1%}', ha='center', va='bottom', fontweight='bold')

ax.set_title('Per-Class Accuracy')
ax.set_ylabel('Accuracy')
ax.set_ylim(0, 1.1)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('per_class_accuracy.png', dpi=150)
plt.show()

## 8. Save Model and Preprocessing Artifacts

Save the trained model, scaler, and label encoder so they can be loaded for game integration.

In [ ]:
# Save model
model_path = 'gesture_model.pth'
torch.save({
    'model_state_dict': model.state_dict(),
    'input_size': input_size,
    'num_classes': num_classes,
    'class_names': list(class_names),
    'test_accuracy': test_accuracy,
}, model_path)
print(f"Model saved to {model_path}")

# Save scaler and label encoder for inference
joblib.dump(scaler, 'gesture_scaler.pkl')
joblib.dump(label_encoder, 'gesture_label_encoder.pkl')
print("Scaler and label encoder saved")

# Print model file size
model_size = os.path.getsize(model_path) / 1024
print(f"Model file size: {model_size:.1f} KB")

## 9. Summary

### Results
- **Model**: MLP with 3 hidden layers (128 → 64 → 32)
- **Input**: 126 features from MediaPipe hand landmarks (position + velocity)
- **Output**: 4 gesture classes (slash, idle, grab, open_palm)
- **Training**: Adam optimizer, CrossEntropy loss, early stopping
- **Hardware**: Apple M3 Pro with MPS acceleration

### How This Connects to the Game
The trained model classifies hand gestures from MediaPipe landmark data in real-time:
- **slash** → triggers fruit cutting in the game
- **grab** → pause/menu interaction
- **open_palm** → start/restart game
- **idle** → no action (hand visible but not performing a gesture)

This creates a complete CV pipeline: Camera → MediaPipe → Gesture Classifier → Game Action

In [ ]:
print("=" * 50)
print("GESTURE CLASSIFIER TRAINING COMPLETE")
print("=" * 50)
print(f"Test Accuracy: {test_accuracy:.1%}")
print(f"Classes: {list(class_names)}")
print(f"Model size: {model_size:.1f} KB")
print(f"Total parameters: {total_params:,}")
print(f"\nSaved files:")
print(f"  - gesture_model.pth (model weights)")
print(f"  - gesture_scaler.pkl (feature scaler)")
print(f"  - gesture_label_encoder.pkl (label mapping)")
print(f"  - training_curves.png")
print(f"  - confusion_matrix.png")
print(f"  - per_class_accuracy.png")
print(f"  - class_distribution.png")